# Concept inconsistency in SkinCon

Rough-set analysis of the SkinCon concepts for the malignant versus non-malignant decision. The notebook downloads the
two SkinCon annotation files from the SkinCon website to `data/skincon`, reads the outputs of `cbm_colab.ipynb` in
`data/colab`, then writes every result to `data/analysis`. It runs on a CPU in a few minutes and needs no image.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"  # the t-SNE layout depends on the number of threads

from pathlib import Path
import math
import urllib.request
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import brentq
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import mutual_info_score, roc_auc_score, roc_curve
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA = Path("data")
SKINCON, COLAB, OUT = DATA / "skincon", DATA / "colab", DATA / "analysis"
OUT.mkdir(parents=True, exist_ok=True)
SKINCON_URL = "https://skincon-dataset.github.io/files"
SKINCON.mkdir(parents=True, exist_ok=True)
for name in ["annotations_fitzpatrick17k.csv", "annotations_ddi.csv"]:
    if not (SKINCON / name).exists():
        urllib.request.urlretrieve(f"{SKINCON_URL}/{name}", SKINCON / name)

## Clean dataset

The Colab notebook cleans the images and assigns the splits. Its prediction file holds one row per clean image and
model, so the rows of the seed-42 hard model give the image list, the split, the diagnosis and the label. The 48
concepts come from the public SkinCon annotation files.

In [2]:
EXCLUDE = "Do not consider this image"


def concept_key(name):
    key = "".join(ch.lower() if ch.isalnum() else "_" for ch in name)
    while "__" in key:
        key = key.replace("__", "_")
    return key.strip("_")


fitz = pd.read_csv(SKINCON / "annotations_fitzpatrick17k.csv")
ddi = pd.read_csv(SKINCON / "annotations_ddi.csv")
names = [c for c in fitz.columns if c not in {"Unnamed: 0", "ImageID", EXCLUDE}]
C = [concept_key(c) for c in names]
fitz["image_id"] = "fitz_" + fitz.ImageID.str.replace(".jpg", "", regex=False)
ddi["image_id"] = "ddi_" + ddi.ImageID.astype(str)
annotations = pd.concat([fitz, ddi])[["image_id"] + names]
annotations.columns = ["image_id"] + C

predictions = pd.read_csv(COLAB / "predictions_efficientnet_b3.csv")
images = predictions[(predictions.seed == 42) & (predictions.model == "hard")]
df = (images[["image_id", "split", "diagnosis", "partition", "source", "y_true"]]
      .merge(annotations, on="image_id", how="left", validate="one_to_one").rename(columns={"y_true": "y"}))
df["dx"] = df.diagnosis.str.replace("-", " ").str.lower().str.strip()
df[C] = df[C].fillna(0).astype(int)
E = pd.read_csv(COLAB / "vocabulary.csv").concept.tolist()   # E24, concepts with at least 30 positive training images

y = df.y.values
tr = (df.split == "train").values
te = (df.split == "test").values
print(f"{len(df)} images, {y.sum()} malignant, {len(C)} concepts, {len(E)} in E24")
df.groupby(["split", "source"]).y.agg(["size", "sum"])

3833 images, 637 malignant, 48 concepts, 24 in E24


size  sum
split source                   
test  ddi              119   31
      fitzpatrick17k   601   94
train ddi              392   96
      fitzpatrick17k  1926  258
valid ddi              120   35
      fitzpatrick17k   675  123

Near-copy groups. Candidate near copies (cosine similarity of at least 0.90 or perceptual hashes within 10 bits) form
the edges of a graph, and every connected component must sit inside one split.

In [3]:
decisions = pd.read_csv(COLAB / "decisions.csv")
pairs = pd.read_csv(COLAB / "candidate_pairs.csv")
kept = decisions[decisions.status == "kept"].copy()
position = {image: i for i, image in enumerate(kept.image_id)}
parent = list(range(len(kept)))


def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a


for a, b in zip(pairs.image_id_a, pairs.image_id_b):
    if a in position and b in position:
        ra, rb = find(position[a]), find(position[b])
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
roots = [find(i) for i in range(len(kept))]
code = {r: k for k, r in enumerate(dict.fromkeys(roots))}
kept["near_copy_group"] = [code[r] for r in roots]
df = df.merge(kept[["image_id", "near_copy_group", "dup_cluster"]], on="image_id", how="left", validate="one_to_one")
crossing = int((df.groupby("near_copy_group").split.nunique() > 1).sum())
print(f"{df.near_copy_group.nunique()} near-copy groups, {crossing} of them cross two splits")

3502 near-copy groups, 0 of them cross two splits


Descriptive malignant groups. They summarize the malignant diagnoses and play no role in training or selection.

In [4]:
GROUP_OF = {
    "melanoma": "Melanoma", "malignant melanoma": "Melanoma", "superficial spreading melanoma ssm": "Melanoma",
    "lentigo maligna": "Melanoma", "melanoma in situ": "Melanoma", "melanoma acral lentiginous": "Melanoma",
    "nodular melanoma (nm)": "Melanoma",
    "basal cell carcinoma": "BCC", "solid cystic basal cell carcinoma": "BCC", "basal cell carcinoma morpheiform": "BCC",
    "basal cell carcinoma nodular": "BCC", "basal cell carcinoma superficial": "BCC",
    "squamous cell carcinoma": "SCC", "squamous cell carcinoma in situ": "SCC",
    "squamous cell carcinoma keratoacanthoma": "SCC",
    "actinic keratosis": "Actinic keratosis",
    "mycosis fungoides": "Cutaneous lymphoma", "subcutaneous t cell lymphoma": "Cutaneous lymphoma",
    "leukemia cutis": "Cutaneous lymphoma", "blastic plasmacytoid dendritic cell neoplasm": "Cutaneous lymphoma",
    "kaposi sarcoma": "Kaposi sarcoma"}
GROUPS = ["BCC", "SCC", "Melanoma", "Cutaneous lymphoma", "Actinic keratosis", "Kaposi sarcoma", "Other"]
SIX = GROUPS[:6]
df["malignant_group"] = [GROUP_OF.get(d, "Other") if v == 1 else None for d, v in zip(df.dx, y)]
group_mask = {g: (df.malignant_group == g).values for g in GROUPS}
df[df.y == 1].malignant_group.value_counts().reindex(GROUPS)

malignant_group
BCC                   189
SCC                   171
Melanoma              126
Cutaneous lymphoma     79
Actinic keratosis      33
Kaposi sarcoma         32
Other                   7
Name: count, dtype: int64

## Rough-set quantities

Images with identical values on a vocabulary form a concept profile. A profile is inconsistent when it holds both
classes, and its images then belong to the boundary region. The ceilings bound the accuracy, the balanced accuracy and
the AUROC of every classifier that reads only the concepts of the vocabulary (Theorems 5 to 7).

In [5]:
def profile_ids(X):
    """Profile index of every row, numbered in order of first appearance."""
    seen, ids = {}, np.empty(len(X), int)
    for i, row in enumerate(np.ascontiguousarray(X.astype(np.uint8))):
        ids[i] = seen.setdefault(row.tobytes(), len(seen))
    return ids


def profile_counts(X, y):
    ids = profile_ids(X)
    k = ids.max() + 1
    n1 = np.bincount(ids, weights=y, minlength=k)
    n0 = np.bincount(ids, minlength=k) - n1
    return ids, n0, n1


def rough_sets(X, y):
    ids, n0, n1 = profile_counts(X, y)
    n = n0 + n1
    mixed = (n0 > 0) & (n1 > 0)
    boundary = mixed[ids]
    P, N = y.sum(), len(y) - y.sum()
    malignant = n1 / P > n0 / N
    rate = n1 / n
    order = np.argsort(rate)
    auc, below = 0.0, 0.0
    for value in np.unique(rate[order]):
        s = order[rate[order] == value]
        pos, neg = n1[s].sum(), n0[s].sum()
        auc += pos * below + 0.5 * pos * neg
        below += neg
    ratio = np.minimum(n0, n1)[mixed] / n[mixed]
    summary = {
        "profiles": int(len(n)), "inconsistent_profiles": int(mixed.sum()), "boundary_images": int(boundary.sum()),
        "gamma": 1 - boundary.mean(), "acc_ceiling": np.maximum(n0, n1).sum() / len(y),
        "ba_ceiling": 0.5 * (n1[malignant].sum() / P + n0[~malignant].sum() / N), "auroc_ceiling": auc / (P * N),
        "malignant_in_boundary": float(boundary[y == 1].mean()), "non_malignant_in_boundary": float(boundary[y == 0].mean()),
        "mean_conflict_ratio": float(ratio.mean()) if len(ratio) else 0, "singleton_profiles": int((n == 1).sum()),
        "singleton_mass": (n == 1).sum() / len(y)}
    return summary, ids, boundary, n0, n1


def entropy(p):
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p)) if 0 < p < 1 else 0.0


def logistic():
    return LogisticRegression(C=0.5, class_weight="balanced", max_iter=3000)


cv20 = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)


def cv_auroc(X):
    return float(np.mean([roc_auc_score(y[b], logistic().fit(X[a], y[a]).decision_function(X[b]))
                          for a, b in cv20.split(X, y)]))


def loo_auroc(X, prior=1.0):
    """AUROC of the malignant rate of the other images of the profile, smoothed towards the prevalence."""
    ids, n0, n1 = profile_counts(X, y)
    return float(roc_auc_score(y, (n1[ids] - y + prior * y.mean()) / ((n0 + n1)[ids] - 1 + prior)))


frequency = df[tr][C].sum().sort_values(ascending=False, kind="stable").index.tolist()
VOCABULARIES = {"A48": C, "E24": E, "V20": frequency[:20], "V12": frequency[:12]}
rows = []
for name, concepts in VOCABULARIES.items():
    X = df[concepts].values
    s, ids, boundary, n0, n1 = rough_sets(X, y)
    n = n0 + n1
    hy = entropy(y.mean())
    hyv = float(sum(n[k] / len(y) * entropy(n1[k] / n[k]) for k in range(len(n))))
    ratio = np.minimum(n0, n1) / n
    hist = np.histogram(ratio[(n0 > 0) & (n1 > 0)], bins=[0, .1, .2, .3, .4, .5001])[0]
    rows.append({"vocabulary": name, "concepts": len(concepts), **s, "label_entropy": hy, "conditional_entropy": hyv,
                 "label_information": (hy - hyv) / hy, "fano_bound": brentq(lambda p: entropy(p) - hyv, 1e-9, .4999),
                 "entropy_bound": hyv / 2, "loo_auroc": loo_auroc(X), "cv_auroc": cv_auroc(X),
                 **{f"conflict_ratio_{lo}_{hi}": int(c) for lo, hi, c in zip(["0", "0.1", "0.2", "0.3", "0.4"],
                                                                              ["0.1", "0.2", "0.3", "0.4", "0.5"], hist)}})
    if name == "E24":
        boundary_e24, ids_e24, n0_e24, n1_e24 = boundary, ids, n0, n1
vocabularies = pd.DataFrame(rows)
vocabularies.to_csv(OUT / "vocabularies.csv", index=False)
vocabularies.set_index("vocabulary").T.round(4)

vocabulary,A48,E24,V20,V12
concepts,48.0000,24.0000,20.0000,12.0000
profiles,867.0000,637.0000,549.0000,277.0000
inconsistent_profiles,114.0000,118.0000,122.0000,93.0000
boundary_images,2469.0000,2753.0000,2891.0000,3390.0000
gamma,0.3559,0.2818,0.2458,0.1156
acc_ceiling,0.8899,0.8800,0.8756,0.8508
ba_ceiling,0.7813,0.7552,0.7436,0.6714
auroc_ceiling,0.8825,0.8539,0.8400,0.7455
malignant_in_boundary,0.6829,0.7488,0.7834,0.9246
non_malignant_in_boundary,0.6364,0.7121,0.7484,0.8764


## Concepts

In [6]:
rows = []
for c, name in zip(C, names):
    x = df[c].values
    k = int(x.sum())
    if k == 0:
        continue
    p = y[x == 1].mean()
    z = 1.96
    d = 1 + z * z / k
    centre = (p + z * z / (2 * k)) / d
    half = z * np.sqrt(p * (1 - p) / k + z * z / (4 * k * k)) / d
    a, b = ((x == 1) & (y == 1)).sum() + .5, ((x == 1) & (y == 0)).sum() + .5
    c_, d_ = ((x == 0) & (y == 1)).sum() + .5, ((x == 0) & (y == 0)).sum() + .5
    rows.append({"concept": c, "name": name, "images": k, "train_images": int(x[tr].sum()), "malignant_rate": p,
                 "rate_low": max(centre - half, 0), "rate_high": min(centre + half, 1), "odds_ratio": a * d_ / (b * c_),
                 "mutual_information_mbit": 1000 * mutual_info_score(x, y) / np.log(2),
                 "boundary_share": x[boundary_e24].mean(), "consistent_share": x[~boundary_e24].mean(),
                 "in_E24": c in E, "in_V20": c in VOCABULARIES["V20"]})
concepts = pd.DataFrame(rows).sort_values("mutual_information_mbit", ascending=False)
concepts.to_csv(OUT / "concepts.csv", index=False)
concepts.round(3)

,concept,name,images,train_images,malignant_rate,rate_low,rate_high,odds_ratio,mutual_information_mbit,boundary_share,consistent_share,in_E24,in_V20
10,crust,Crust,548,321,0.330,0.292,0.371,3.061,20.023,0.118,0.206,True,True
19,telangiectasia,Telangiectasia,104,53,0.606,0.510,0.694,8.404,19.886,0.016,0.055,True,True
34,black,Black,118,59,0.517,0.428,0.605,5.825,14.982,0.026,0.043,True,True
9,ulcer,Ulcer,167,99,0.371,0.302,0.447,3.182,8.042,0.024,0.094,True,True
30,translucent,Translucent,23,14,0.783,0.581,0.903,17.329,8.040,0.004,0.010,False,False
5,pustule,Pustule,101,62,0.000,0.000,0.037,0.024,7.011,0.000,0.094,True,True
39,umbilicated,Umbilicated,54,30,0.519,0.389,0.646,5.594,6.749,0.005,0.037,True,False
6,bulla,Bulla,64,36,0.000,0.000,0.057,0.038,4.419,0.000,0.059,True,False
46,pigmented,Pigmented,6,4,1.000,0.610,1.000,65.803,4.062,0.001,0.003,False,False
8,nodule,Nodule,234,143,0.278,0.224,0.338,2.044,3.689,0.020,0.167,True,True


Forced errors by diagnosis. The rule that attains the balanced accuracy ceiling labels a profile malignant when
$n_k^1/P > n_k^0/N$. Images that it misclassifies are forced errors, and malignant forced errors are forced misses.

In [7]:
balanced_rule = (n1_e24 / y.sum() > n0_e24 / (len(y) - y.sum()))[ids_e24]
t = pd.DataFrame({"diagnosis": df.dx, "y": y, "boundary": boundary_e24,
                  "forced_error": balanced_rule != (y == 1)})


def by_diagnosis(part):
    return part.groupby("diagnosis").agg(images=("y", "size"), boundary_share=("boundary", "mean"),
                                         forced_error_rate=("forced_error", "mean"))


fp_dx = by_diagnosis(t[t.y == 0])
fp_dx = fp_dx[fp_dx.images >= 20].sort_values("forced_error_rate", ascending=False)
fn_dx = by_diagnosis(t[t.y == 1])
fn_dx = fn_dx[fn_dx.images >= 15].sort_values("forced_error_rate", ascending=False)
fp_dx.to_csv(OUT / "forced_false_positives_by_diagnosis.csv")
fn_dx.to_csv(OUT / "forced_misses_by_diagnosis.csv")
fn_dx.round(3)

,images,boundary_share,forced_error_rate
diagnosis,,,
kaposi sarcoma,32,0.875,0.438
mycosis fungoides,76,0.961,0.434
squamous cell carcinoma in situ,25,0.880,0.400
malignant melanoma,18,0.667,0.278
basal cell carcinoma,154,0.688,0.266
actinic keratosis,33,0.758,0.242
melanoma,58,0.741,0.224
superficial spreading melanoma ssm,17,0.882,0.176
lentigo maligna,22,0.864,0.136


## Shapley attribution

Shapley values of every E24 concept for the quality of classification $\gamma$, the balanced accuracy ceiling and,
per malignant group, the detectability (the share of the group in profiles that the balanced rule labels malignant).
They are estimated from 3,000 random permutations.

In [8]:
X = df[E].values.astype(np.int64)
n_img = len(y)
P, N = y.sum(), n_img - y.sum()
present = [g for g in GROUPS if group_mask[g].any()]


def value_functions(codes):
    _, inverse = np.unique(codes, return_inverse=True)
    k = inverse.max() + 1
    n1 = np.bincount(inverse, weights=y, minlength=k)
    n0 = np.bincount(inverse, minlength=k) - n1
    consistent = ~((n0 > 0) & (n1 > 0))[inverse]
    malignant = n1 / P > n0 / N
    detected = malignant[inverse]
    out = [consistent.mean(), 0.5 * (n1[malignant].sum() / P + n0[~malignant].sum() / N)]
    for g in present:
        out += [detected[group_mask[g]].mean(), consistent[group_mask[g]].mean()]
    return np.array(out)


rng = np.random.default_rng(0)
R, d = 3000, len(E)
phi = np.zeros((d, 2 + 2 * len(present)))
phi2 = np.zeros_like(phi)
empty = value_functions(np.zeros(n_img, dtype=np.int64))
for _ in range(R):
    codes, previous = np.zeros(n_img, dtype=np.int64), empty
    for j in rng.permutation(d):
        codes = codes | (X[:, j] << j)
        current = value_functions(codes)
        gain = current - previous
        phi[j] += gain
        phi2[j] += gain * gain
        previous = current
phi /= R
se = np.sqrt(np.maximum(phi2 / R - phi ** 2, 0) / R)
columns = ["gamma", "ba"] + [f"{kind}|{g}" for g in present for kind in ("det", "cons")]
shapley = pd.DataFrame(phi, index=pd.Index(E, name="concept"), columns=columns)
shapley.to_csv(OUT / "shapley.csv")
pd.DataFrame(se, index=pd.Index(E, name="concept"), columns=columns).to_csv(OUT / "shapley_standard_errors.csv")
shapley[["gamma", "ba"] + [f"det|{g}" for g in SIX]].round(3)

,gamma,ba,det|BCC,det|SCC,det|Melanoma,det|Cutaneous lymphoma,det|Actinic keratosis,det|Kaposi sarcoma
concept,,,,,,,,
erythema,0.016,0.010,0.051,0.048,0.061,0.066,0.028,0.062
plaque,0.019,0.009,0.056,0.064,0.032,0.111,0.027,0.068
papule,0.019,0.011,0.037,0.069,0.059,0.135,0.039,0.020
brown_hyperpigmentation,0.016,0.021,0.022,0.007,0.192,0.138,0.063,0.166
scale,0.016,0.007,0.073,0.069,0.075,0.020,0.084,0.108
crust,0.020,0.051,0.077,0.160,-0.025,-0.046,0.309,-0.070
white_hypopigmentation,0.016,0.007,0.093,0.073,0.103,0.094,0.085,0.103
yellow,0.011,0.008,-0.025,0.017,-0.026,-0.031,0.080,-0.018
nodule,0.010,0.008,-0.024,0.016,0.005,-0.022,-0.028,0.006


Conflict participation, the share of the conflict mass $\sum_k \min(n_k^0, n_k^1)$ that involves a concept,
compared with its prevalence.

In [9]:
ids, n0, n1 = profile_counts(X, y)
nk = n0 + n1
mixed = (n0 > 0) & (n1 > 0)
conflict = np.minimum(n0, n1)
rows = []
for j, c in enumerate(E):
    has = np.array([X[ids == k][0, j] for k in range(len(nk))])
    rows.append({"concept": c, "prevalence": X[:, j].mean(), "conflict_share": conflict[has == 1].sum() / conflict.sum(),
                 "boundary_share": nk[mixed & (has == 1)].sum() / nk[mixed].sum()})
participation = pd.DataFrame(rows)
participation.to_csv(OUT / "conflict_participation.csv", index=False)
participation.assign(ratio=participation.conflict_share / participation.prevalence).sort_values("ratio", ascending=False).round(3)

,concept,prevalence,conflict_share,boundary_share,ratio
17,black,0.031,0.065,0.026,2.118
5,crust,0.143,0.200,0.118,1.399
3,brown_hyperpigmentation,0.291,0.365,0.316,1.253
10,erosion,0.056,0.067,0.037,1.213
19,telangiectasia,0.027,0.033,0.016,1.202
13,ulcer,0.044,0.052,0.024,1.198
12,friable,0.044,0.048,0.025,1.098
1,plaque,0.553,0.598,0.576,1.082
14,exudate,0.041,0.043,0.023,1.061
4,scale,0.204,0.215,0.207,1.056


## Forced misses and the collisions behind them

In [10]:
malignant_profile = n1 / P > n0 / N
forced_miss = (y == 1) & ~malignant_profile[ids]
df["profile"] = ids
df["forced_miss"] = forced_miss
df["in_boundary"] = mixed[ids]

collisions, signatures = [], []
for g in present:
    m = group_mask[g] & forced_miss
    for k in np.unique(ids[m]):
        members = df[ids == k]
        missed = int((m & (ids == k)).sum())
        negatives = members[members.y == 0]
        for dx, count in negatives.dx.value_counts().items():
            collisions.append({"malignant_group": g, "diagnosis": dx, "weight": missed * count / len(negatives)})
        signatures.append({
            "malignant_group": g,
            "profile": ", ".join(c.replace("_", " ") for c in E if members[E].iloc[0][c]) or "(no concept)",
            "forced_misses": missed, "images": len(members), "malignant": int(members.y.sum()),
            "top_non_malignant": ", ".join(negatives.dx.value_counts().head(3).index)})
(pd.DataFrame(collisions).groupby(["malignant_group", "diagnosis"]).weight.sum().unstack(fill_value=0)
 .to_csv(OUT / "collisions.csv"))
signatures = pd.DataFrame(signatures).sort_values(["malignant_group", "forced_misses"], ascending=[True, False])
signatures.to_csv(OUT / "forced_miss_profiles.csv", index=False)

ids48, n0_48, n1_48 = profile_counts(df[C].values, y)
forced_48 = ~(n1_48 / P > n0_48 / N)[ids48]
groups = pd.DataFrame([{"malignant_group": g, "images": int(group_mask[g].sum()), "test_images": int((group_mask[g] & te).sum()),
                        "boundary_share": df.in_boundary[group_mask[g]].mean(), "forced_miss_share": forced_miss[group_mask[g]].mean(),
                        "forced_miss_share_A48": (forced_48 & group_mask[g]).sum() / group_mask[g].sum()} for g in present])
groups.to_csv(OUT / "malignant_groups.csv", index=False)
groups.round(3)

,malignant_group,images,test_images,boundary_share,forced_miss_share,forced_miss_share_A48
0,BCC,189,42,0.693,0.254,0.212
1,SCC,171,33,0.655,0.170,0.152
2,Melanoma,126,20,0.794,0.222,0.214
3,Cutaneous lymphoma,79,13,0.949,0.443,0.392
4,Actinic keratosis,33,5,0.758,0.242,0.303
5,Kaposi sarcoma,32,9,0.875,0.438,0.438
6,Other,7,3,0.857,0.571,0.571


Single concept edits. An edit switches one concept of a forced miss, and it rescues the image when the new profile
exists in the data and the balanced rule labels it malignant.

In [11]:
lookup = {tuple(X[ids == k][0]): k for k in range(len(n0))}
group_of_image = df.set_index("image_id").malignant_group
rows = []
for i in np.where(forced_miss)[0]:
    v = X[i]
    for j, c in enumerate(E):
        w = v.copy()
        w[j] = 1 - w[j]
        k = lookup.get(tuple(w))
        rows.append({"image_id": df.image_id[i], "malignant_group": group_of_image[df.image_id[i]], "concept": c,
                     "edit": "add" if v[j] == 0 else "drop",
                     "status": "no data" if k is None else ("malignant" if malignant_profile[k] else "non-malignant")})
edits = pd.DataFrame(rows)
edits.to_csv(OUT / "concept_edits.csv", index=False)
rescued = edits[edits.status == "malignant"]
rescues = rescued.groupby(["malignant_group", "concept", "edit"]).image_id.nunique().reset_index()
rescues["share"] = rescues.image_id / rescues.malignant_group.map(edits.groupby("malignant_group").image_id.nunique())
rescues = rescues.rename(columns={"image_id": "images"})
rescues.to_csv(OUT / "rescue_edits.csv", index=False)
(rescued[rescued.edit == "add"].groupby("concept").image_id.nunique() / edits.image_id.nunique()).sort_values(ascending=False).head(8).round(3)

concept
crust                      0.825
black                      0.669
telangiectasia             0.608
brown_hyperpigmentation    0.458
umbilicated                0.367
erosion                    0.355
ulcer                      0.343
purple                     0.265
Name: image_id, dtype: float64

## Filtering experiment

Five filters act on training images only. Symmetric removal drops every boundary image, asymmetric removal drops the
non-malignant boundary images, majority editing drops the minority class of every inconsistent profile, weighted editing
does the same with class-weighted counts, and $\tau$-editing restricts majority editing to profiles with a conflict ratio
of at most 0.2 and at least five images. A gradient boosting classifier on the E24 concepts is trained after each
filter and always evaluated on unfiltered images.

In [12]:
FILTERS = ["none", "symmetric", "asymmetric", "majority", "weighted", "tau"]


def keep_mask(Xs, ys, how, tau=0.2, min_size=5):
    i, a0, a1 = profile_counts(Xs, ys)
    size = a0 + a1
    boundary = ((a0 > 0) & (a1 > 0))[i]
    ratio = np.minimum(a0, a1) / size
    if how == "none":
        return np.ones(len(ys), bool)
    if how == "symmetric":
        return ~boundary
    if how == "asymmetric":
        return ~boundary | (ys == 1)
    if how in ("majority", "tau"):
        majority = (a1 > a0)[i]
    else:
        majority = (a1 / ys.sum() > a0 / (len(ys) - ys.sum()))[i]
    minority = boundary & (ys != majority)
    if how == "tau":
        minority &= (ratio[i] <= tau) & (size[i] >= min_size)
    return ~minority


def booster():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, class_weight="balanced",
                                          random_state=0)


U, ids_u = np.unique(X, axis=0, return_inverse=True)
ids_u = ids_u.reshape(-1)
images_u = np.bincount(ids_u)
malignant_u = np.bincount(ids_u, weights=y)
maps, oof, retention, training = {}, {}, [], []
for how in FILTERS:
    k = keep_mask(X[tr], y[tr], how)
    maps[how] = booster().fit(X[tr][k], y[tr][k]).predict_proba(U)[:, 1]
    for g in present:
        retention.append({"filter": how, "group": g, "retained": (k & group_mask[g][tr]).sum() / group_mask[g][tr].sum()})
    retention.append({"filter": how, "group": "Non-malignant", "retained": (k & (y[tr] == 0)).sum() / (y[tr] == 0).sum()})
    score, count = np.zeros(n_img), np.zeros(n_img)
    for repeat in range(4):
        for a, b in StratifiedKFold(5, shuffle=True, random_state=repeat).split(X, y):
            kk = keep_mask(X[a], y[a], how)
            score[b] += booster().fit(X[a][kk], y[a][kk]).predict_proba(X[b])[:, 1]
            count[b] += 1
    oof[how] = score / count
    filtered = []
    for a, b in RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=1).split(X, y):
        kk = keep_mask(X[a], y[a], how)
        s = booster().fit(X[a][kk], y[a][kk]).predict_proba(X[b])[:, 1]
        ke = keep_mask(X[b], y[b], how)
        filtered.append(roc_auc_score(y[b][ke], s[ke]) if 0 < y[b][ke].sum() < ke.sum() else np.nan)
    training.append({"filter": how, "train_images_kept": int(k.sum()), "train_malignant_kept": int(y[tr][k].sum()),
                     "train_gamma": rough_sets(X[tr][k], y[tr][k])[0]["gamma"], "filtered_cv_auroc": float(np.nanmean(filtered))})
pd.DataFrame(retention).to_csv(OUT / "filter_retention.csv", index=False)
filter_training = pd.DataFrame(training)
filter_training.to_csv(OUT / "filter_training.csv", index=False)

rows = []
for how in FILTERS:
    called = oof[how] >= 0.5
    row = {"filter": how, "auroc": roc_auc_score(y, oof[how]), "sensitivity": called[y == 1].mean(),
           "specificity": (~called)[y == 0].mean()}
    row.update({g: called[group_mask[g]].mean() for g in present})
    rows.append(row)
filter_oof = pd.DataFrame(rows)
filter_oof.to_csv(OUT / "filter_out_of_fold.csv", index=False)
filter_training.merge(filter_oof, on="filter").round(3)

,filter,train_images_kept,train_malignant_kept,train_gamma,filtered_cv_auroc,auroc,sensitivity,specificity,BCC,SCC,Melanoma,Cutaneous lymphoma,Actinic keratosis,Kaposi sarcoma,Other
0,none,2318,354,0.312,0.754,0.749,0.626,0.754,0.630,0.719,0.675,0.456,0.697,0.312,0.429
1,symmetric,724,104,1.000,0.891,0.710,0.438,0.881,0.524,0.556,0.437,0.076,0.515,0.125,0.429
2,asymmetric,974,354,1.000,0.814,0.661,0.882,0.291,0.899,0.895,0.929,0.848,0.848,0.656,0.857
3,majority,2073,117,1.000,0.924,0.711,0.377,0.906,0.450,0.444,0.452,0.063,0.273,0.156,0.429
4,weighted,1780,266,1.000,0.843,0.738,0.651,0.723,0.630,0.737,0.754,0.481,0.697,0.344,0.429
5,tau,2188,224,0.822,0.826,0.733,0.542,0.808,0.582,0.678,0.556,0.241,0.606,0.219,0.429


Decision maps. The 637 E24 profiles are placed by t-SNE on their Hamming distances, each with the probability of the
model trained on the filtered training split.

In [13]:
hamming = (U[:, None, :] != U[None, :, :]).sum(-1).astype(float)
layout = TSNE(n_components=2, metric="precomputed", init="random", perplexity=40, random_state=3,
              max_iter=3000).fit_transform(hamming)
layout = (layout - layout.mean(0)) / layout.std(0)
profile_maps = pd.DataFrame(U, columns=E)
profile_maps["images"] = images_u
profile_maps["malignant"] = malignant_u
profile_maps["tsne_x"], profile_maps["tsne_y"] = layout[:, 0], layout[:, 1]
for how in FILTERS:
    profile_maps[f"map_{how}"] = maps[how]
profile_maps.to_csv(OUT / "profile_maps.csv", index_label="profile")
scores = pd.DataFrame({"image_id": df.image_id, "y": y, "test": te, "profile": ids_u, **{f"oof_{h}": oof[h] for h in FILTERS}})
scores.to_csv(OUT / "out_of_fold_scores.csv", index=False)
df[["image_id", "source", "split", "near_copy_group", "dup_cluster", "diagnosis", "dx", "partition", "y",
    "malignant_group", "profile", "forced_miss", "in_boundary"]].to_csv(OUT / "clean_dataset.csv", index=False)
profile_maps[["images", "malignant", "tsne_x", "tsne_y", "map_none", "map_majority", "map_asymmetric"]].head()

,images,malignant,tsne_x,tsne_y,map_none,map_majority,map_asymmetric
0,9,0.0,0.471494,0.531927,0.355497,0.035333,0.659223
1,4,0.0,0.581405,0.458413,0.016051,0.011267,0.028332
2,1,1.0,0.450657,0.492841,0.801726,0.707310,0.844947
3,7,3.0,0.411324,0.596141,0.861547,0.928420,0.804517
4,3,0.0,1.546157,-0.297002,0.081173,0.017331,0.084418


## Concept bottleneck models

The Colab notebook trains hard, soft and hybrid concept bottleneck models with three seeds. This section evaluates
their test predictions against the rough-set quantities of the annotated concepts.

In [14]:
image_info = df.set_index("image_id")
predictions["malignant_group"] = predictions.image_id.map(image_info.malignant_group)
predictions["forced"] = predictions.image_id.map(image_info.forced_miss)
MODELS = ["hard", "soft", "hybrid"]


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(centre - half, 0), min(centre + half, 1)


train = df.split == "train"
annotated_model = logistic().fit(df.loc[train, E].values, y[train.values])
df["annotated_logit"] = annotated_model.decision_function(df[E].values)
df["annotated_pred"] = (df.annotated_logit > 0).astype(int)
predictions["annotated_pred"] = predictions.image_id.map(df.set_index("image_id").annotated_pred)
test = predictions[predictions.split == "test"]
valid = predictions[predictions.split == "valid"]

curves = []
for m in MODELS:
    for s in sorted(test.seed.unique()):
        t = test[(test.model == m) & (test.seed == s)]
        f, tp, _ = roc_curve(t.y_true, t.logit)
        curves.append(pd.DataFrame({"curve": f"{m}_{s}", "fpr": f, "tpr": tp}))
test_images = df[df.split == "test"]
f, tp, _ = roc_curve(test_images.y, test_images.annotated_logit)
curves.append(pd.DataFrame({"curve": "annotated", "fpr": f, "tpr": tp}))
ids_t, n0_t, n1_t = profile_counts(test_images[E].values, test_images.y.values)
rate_t = (n1_t / (n0_t + n1_t))[ids_t]
f, tp, _ = roc_curve(test_images.y, rate_t)
curves.append(pd.DataFrame({"curve": "ceiling", "fpr": f, "tpr": tp}))
pd.concat(curves).to_csv(OUT / "roc_curves.csv", index=False)
baselines = pd.DataFrame({"quantity": ["annotated_concepts_test_auroc", "ceiling_test_auroc"],
                          "value": [roc_auc_score(test_images.y, test_images.annotated_logit), roc_auc_score(test_images.y, rate_t)]})
baselines.to_csv(OUT / "concept_baselines.csv", index=False)
baselines

,quantity,value
0,annotated_concepts_test_auroc,0.755267
1,ceiling_test_auroc,0.909526


Miss rates per malignant group at the default threshold, with 95% Wilson intervals over the pooled seeds.

In [15]:
rows = []
for g in present + ["All malignant"]:
    mt = test[(test.y_true == 1) & ((test.malignant_group == g) | (g == "All malignant"))]
    row = {"malignant_group": g, "test_images": mt.image_id.nunique()}
    for m in MODELS:
        p = mt[mt.model == m]
        lo, hi = wilson((p.y_pred == 0).sum(), len(p))
        row[m] = 1 - p.groupby("seed").y_pred.mean().mean()
        row[m + "_low"], row[m + "_high"] = lo, hi
    first = mt.drop_duplicates("image_id")
    row["annotated"] = 1 - first.annotated_pred.mean()
    row["forced"] = first.forced.mean()
    rows.append(row)
miss_rates = pd.DataFrame(rows)
miss_rates.to_csv(OUT / "miss_rates.csv", index=False)
miss_rates.round(3)

,malignant_group,test_images,hard,hard_low,hard_high,soft,soft_low,soft_high,hybrid,hybrid_low,hybrid_high,annotated,forced
0,BCC,42,0.143,0.092,0.215,0.143,0.092,0.215,0.119,0.073,0.187,0.405,0.214
1,SCC,33,0.152,0.094,0.235,0.141,0.086,0.223,0.152,0.094,0.235,0.273,0.152
2,Melanoma,20,0.267,0.171,0.390,0.217,0.131,0.336,0.217,0.131,0.336,0.450,0.300
3,Cutaneous lymphoma,13,0.436,0.293,0.590,0.436,0.293,0.590,0.462,0.316,0.614,0.538,0.308
4,Actinic keratosis,5,0.533,0.301,0.752,0.533,0.301,0.752,0.533,0.301,0.752,0.600,0.400
5,Kaposi sarcoma,9,0.407,0.245,0.593,0.370,0.215,0.558,0.407,0.245,0.593,0.556,0.444
6,Other,3,0.444,0.189,0.733,0.333,0.121,0.646,0.444,0.189,0.733,0.667,0.667
7,All malignant,125,0.237,0.197,0.283,0.221,0.182,0.266,0.224,0.185,0.269,0.416,0.256


Operating points. Thresholds for 90% and 95% sensitivity are set on the validation images of each seed.

In [16]:
points, flags = [], []
for target in (0.0, 0.90, 0.95):
    for m in MODELS:
        for s in sorted(test.seed.unique()):
            t = test[(test.model == m) & (test.seed == s)]
            if target == 0:
                threshold = 0.0
                called = t.logit > 0
            else:
                v = valid[(valid.model == m) & (valid.seed == s)]
                logits = np.sort(v[v.y_true == 1].logit.values)
                threshold = logits[int(np.floor((1 - target) * len(logits)))]
                called = t.logit >= threshold
            row = {"target": target, "model": m, "seed": s, "threshold": threshold,
                   "sensitivity": called[t.y_true == 1].mean(), "specificity": (~called)[t.y_true == 0].mean(),
                   "flagged": called.mean()}
            for part in ["benign", "non-neoplastic"]:
                row["fpr_" + part] = called[(t.y_true == 0) & (t.partition == part)].mean()
            points.append(row)
            if target == 0.95:
                negatives = t[t.y_true == 0].assign(called=called[t.y_true == 0].values)
                for dx, g in negatives.groupby(negatives.image_id.map(image_info.dx)):
                    flags.append({"model": m, "seed": s, "diagnosis": dx, "test_images": len(g), "flag_rate": g.called.mean()})
operating_points = pd.DataFrame(points)
operating_points.to_csv(OUT / "operating_points.csv", index=False)
flagged = (pd.DataFrame(flags).groupby(["model", "diagnosis"])
           .agg(test_images=("test_images", "first"), flag_rate=("flag_rate", "mean")).reset_index())
flagged = flagged[flagged.test_images >= 5]
flagged.to_csv(OUT / "flagged_diagnoses.csv", index=False)
operating_points.groupby(["target", "model"])[["sensitivity", "specificity"]].agg(["mean", "std"]).round(3)

sensitivity        specificity       
                     mean    std        mean    std
target model                                       
0.00   hard         0.763  0.036       0.675  0.060
       hybrid       0.776  0.071       0.687  0.086
       soft         0.779  0.040       0.726  0.021
0.90   hard         0.925  0.023       0.424  0.038
       hybrid       0.915  0.023       0.475  0.074
       soft         0.912  0.032       0.528  0.060
0.95   hard         0.949  0.024       0.328  0.054
       hybrid       0.963  0.028       0.292  0.064
       soft         0.957  0.020       0.362  0.067

Hybrid model. The logit is the sum of a concept part $f$ and a visual part $g$ with $|g| \le \|w\|_1$ (Lemma 17).

In [17]:
hybrid = test[test.model == "hybrid"][["seed", "image_id", "y_true", "concept_part", "visual_part", "y_pred",
                                       "malignant_group", "logit"]]
hard = test[test.model == "hard"][["seed", "image_id", "y_pred"]].rename(columns={"y_pred": "hard_pred"})
hybrid = hybrid.merge(hard, on=["seed", "image_id"])
hybrid.to_csv(OUT / "hybrid_parts.csv", index=False)
metrics = pd.read_csv(COLAB / "training_metrics.csv")
weights = metrics[(metrics.model == "hybrid") & (metrics.split == "test") & (metrics.source == "all")][["seed", "w_l1"]]
weights.to_csv(OUT / "hybrid_weights.csv", index=False)
hybrid["changed"] = np.sign(hybrid.concept_part) != np.sign(hybrid.logit)
changed = hybrid[hybrid.changed].assign(abs_f=lambda h: h.concept_part.abs())
changed.groupby("seed").agg(decisions_changed=("abs_f", "size"), largest_abs_f=("abs_f", "max")).join(weights.set_index("seed"))

,decisions_changed,largest_abs_f,w_l1
seed,,,
42,32,0.314392,0.576025
64,2,0.001528,0.020813
128,81,0.355555,0.742567


Forced cases among the test malignancies. A forced case is a malignant image whose annotated profile the balanced
rule labels non-malignant.

In [18]:
malignant_test = test[test.y_true == 1].assign(missed=lambda t: t.y_pred == 0)
outcomes = (malignant_test.assign(forced_missed=lambda t: t.missed & t.forced, forced_detected=lambda t: ~t.missed & t.forced,
                                  other_missed=lambda t: t.missed & ~t.forced, other_detected=lambda t: ~t.missed & ~t.forced)
            .groupby(["model", "seed"])[["forced_missed", "forced_detected", "other_missed", "other_detected"]].sum().reset_index())
outcomes.to_csv(OUT / "forced_case_outcomes.csv", index=False)
outcomes.groupby("model").mean(numeric_only=True).drop(columns="seed")

,forced_missed,forced_detected,other_missed,other_detected
model,,,,
hard,13.000000,19.000000,16.666667,76.333333
hybrid,12.666667,19.333333,15.333333,77.666667
soft,12.333333,19.666667,15.333333,77.666667


## Values quoted in the article

Every number of the text, recomputed from the files above and from the Colab outputs.

In [19]:
reported = []


def report(section, quantity, value):
    reported.append({"section": section, "quantity": quantity,
                     "value": value.item() if hasattr(value, "item") else value})


counts = pd.read_csv(COLAB / "clean_counts.csv").set_index("quantity").value
settings = pd.read_csv(COLAB / "cleaning_settings.csv").set_index("setting").value
by_threshold = pd.read_csv(COLAB / "duplicate_pairs_by_threshold.csv").set_index("threshold")
reliability = pd.read_csv(COLAB / "concept_reliability_duplicates.csv").set_index("concept")
removal = pd.read_csv(COLAB / "removal_by_subgroup.csv")
f17kc = pd.read_csv(COLAB / "audit_fitzpatrick17k_c.csv").set_index("status")
complexity = pd.read_csv(COLAB / "complexity_test.csv")
similarity = pd.read_csv(COLAB / "complexity_similarities.csv").set_index("pairs")
summary = pd.read_csv(COLAB / "summary_test.csv")
summary = summary[summary.source == "all"].set_index("model")
voc = vocabularies.set_index("vocabulary")
grp = groups.set_index("malignant_group")
n_mal = int(y.sum())

section = "Clean dataset"
report(section, "Fitzpatrick17k images annotated by SkinCon", counts["fitzpatrick17k_annotated"])
report(section, "DDI images annotated by SkinCon", counts["ddi_annotated"])
report(section, "Fitzpatrick17k images flagged by the annotators", int(fitz[EXCLUDE].fillna(0).astype(bool).sum()))
report(section, "DDI images flagged by the annotators", int(ddi[EXCLUDE].fillna(0).astype(bool).sum()))
report(section, "Fitzpatrick17k images excluded by quality control",
       counts["fitzpatrick17k_after_exclusion_flag"] - counts["fitzpatrick17k_after_quality_control"])
report(section, "images entering the cleaning rules", counts["images_found"])
report(section, "candidate near-copy pairs", counts["candidate_pairs"])
report(section, "confirmed duplicate pairs", counts["confirmed_duplicate_pairs"])
confirmed = pairs[pairs.confirmed == True]
both = (confirmed.cosine >= float(settings["tau_duplicate"])) & (confirmed.phash_bits <= int(settings["phash_confirm_bits"]))
report(section, "confirmed pairs that satisfy both detectors", int(both.sum()))
for tau in (0.9, 0.95):
    report(section, f"pairs with cosine of at least {tau}", int(by_threshold.loc[tau, "pairs"]))
    report(section, f"pairs with cosine of at least {tau} that differ in diagnosis", int(by_threshold.loc[tau, "diagnosis differs (D)"]))
    report(section, f"pairs with cosine of at least {tau} that differ in malignancy", int(by_threshold.loc[tau, "malignancy differs"]))
report(section, "unconfirmed pairs left for visual review", counts["unconfirmed_pairs_for_review"])
if "near_copy_pairs_across_earlier_split" in counts:
    report(section, "near-copy pairs across the splits of the earlier partition", counts["near_copy_pairs_across_earlier_split"])
report(section, "share of images removed", counts["duplicates_removed"] / counts["images_found"])
report(section, "removal rate of malignant images", removal[(removal.attribute == "malignant") & (removal.level.astype(str) == "1")].removal_rate.iloc[0])
report(section, "removal rate of non-malignant images", removal[(removal.attribute == "malignant") & (removal.level.astype(str) == "0")].removal_rate.iloc[0])
report(section, "chi-square p-value of the removal rate, malignancy", removal[removal.attribute == "malignant"].chi2_p_value.iloc[0])
report(section, "chi-square p-value of the removal rate, skin type", removal[removal.attribute == "skin_type"].chi2_p_value.iloc[0])
report(section, "Fitzpatrick17k images kept", int(f17kc.sum(axis=1)["kept"]))
report(section, "kept Fitzpatrick17k images that belong to Fitzpatrick17k-C", int(f17kc.loc["kept", "in F17k-C"]))
report(section, "removed Fitzpatrick17k duplicates", int(f17kc.loc["duplicate_removed"].sum()))
report(section, "removed duplicates also excluded from Fitzpatrick17k-C", int(f17kc.loc["duplicate_removed", "not in F17k-C"]))
report(section, "clean images", n_img)
report(section, "malignant images", n_mal)
report(section, "prevalence of malignancy", y.mean())
for part, count in df.partition.value_counts().items():
    report(section, f"{part} images", count)
report(section, "diagnoses", df.dx.nunique())
report(section, "malignant diagnosis labels", df[df.y == 1].dx.nunique())
for s in ["train", "valid", "test"]:
    report(section, f"{s} images", int((df.split == s).sum()))
    report(section, f"malignant {s} images", int(df[df.split == s].y.sum()))
for g in present:
    report(section, f"{g} images", int(grp.images[g]))
report(section, "minimum number of positive training images for E24", int(settings["min_train_positives"]))

section = "Inconsistency"
e24 = voc.loc["E24"]
report(section, "E24 boundary images", int(e24.boundary_images))
report(section, "E24 boundary share", e24.boundary_images / n_img)
report(section, "E24 accuracy ceiling", e24.acc_ceiling)
report(section, "E24 balanced accuracy ceiling", e24.ba_ceiling)
report(section, "E24 AUROC ceiling", e24.auroc_ceiling)
report(section, "V20 AUROC ceiling", voc.loc["V20", "auroc_ceiling"])
report(section, "accuracy of the rule that labels every image non-malignant", 1 - y.mean())
report(section, "A48 boundary share", voc.loc["A48", "boundary_images"] / n_img)
for v in ["A48", "E24", "V12"]:
    report(section, f"{v} label information", voc.loc[v, "label_information"])
report(section, "E24 lower bound on the error (Fano)", e24.fano_bound)
report(section, "E24 profiles", int(e24.profiles))
order = np.argsort(-nk, kind="stable")
report(section, "share of the boundary held by the 10 largest profiles", nk[order[:10]][mixed[order[:10]]].sum() / df.in_boundary.sum())
report(section, "lowest malignant rate among the 10 largest profiles", (n1 / nk)[order[:10]].min())
report(section, "highest malignant rate among the 10 largest profiles", (n1 / nk)[order[:10]].max())
for rank, k in enumerate(order[:4], 1):
    members = df[ids == k]
    label = " with ".join(c.replace("_", " ") for c in E if members[c].iloc[0])
    report(section, f"images in the profile of rank {rank} ({label})", int(nk[k]))
    report(section, f"diagnoses in the profile of rank {rank}", members.dx.nunique())
ratio = (np.minimum(n0, n1) / nk)[mixed]
report(section, "inconsistent profiles", int(mixed.sum()))
report(section, "inconsistent profiles with a conflict ratio of at most 0.2", int((ratio <= 0.2).sum()))
report(section, "inconsistent profiles with a conflict ratio of one half", int((ratio == 0.5).sum()))
report(section, "malignant images in the boundary", df.in_boundary[df.y == 1].mean())
report(section, "non-malignant images in the boundary", df.in_boundary[df.y == 0].mean())
for g in SIX:
    report(section, f"{g} images in the boundary", grp.boundary_share[g])
benign, neoplastic = (df.partition == "benign").values, (df.partition == "non-neoplastic").values
report(section, "benign images in the boundary", df.in_boundary[benign].mean())
report(section, "non-neoplastic images in the boundary", df.in_boundary[neoplastic].mean())
forced_error = balanced_rule != (y == 1)
for g in SIX:
    report(section, f"forced errors among {g} images", forced_error[group_mask[g]].mean())
report(section, "forced errors among benign images", forced_error[benign].mean())
report(section, "forced errors among non-neoplastic images", forced_error[neoplastic].mean())

section = "Concepts"
cpt = concepts.set_index("concept")
for c in ["erythema", "plaque", "papule"]:
    report(section, f"images with {c}", cpt.images[c])
    report(section, f"mutual information of {c} (mbit)", cpt.mutual_information_mbit[c])
for c in ["crust", "telangiectasia", "black"]:
    report(section, f"mutual information of {c} (mbit)", cpt.mutual_information_mbit[c])
for c in ["crust", "black", "telangiectasia", "brown_hyperpigmentation"]:
    report(section, f"Shapley value of {c} for the balanced accuracy ceiling", shapley.ba[c])
part_ = participation.set_index("concept")
enrichment = part_.conflict_share / part_.prevalence
report(section, "share of the conflict mass that involves black", part_.conflict_share["black"])
report(section, "prevalence of black", part_.prevalence["black"])
for c in ["black", "crust", "brown_hyperpigmentation", "erosion", "telangiectasia", "ulcer"]:
    report(section, f"conflict enrichment of {c}", enrichment[c])
for c, g in [("crust", "SCC"), ("crust", "Actinic keratosis"), ("brown_hyperpigmentation", "Melanoma"),
             ("brown_hyperpigmentation", "Kaposi sarcoma"), ("brown_hyperpigmentation", "Cutaneous lymphoma"),
             ("black", "Kaposi sarcoma"), ("black", "Cutaneous lymphoma")]:
    report(section, f"Shapley value of {c} for the detectability of {g}", shapley[f"det|{g}"][c])
benign_markers = shapley.loc[["white_hypopigmentation", "pustule", "patch"], [f"det|{g}" for g in SIX]].values
report(section, "smallest detectability value of white hypopigmentation, pustule and patch", benign_markers.min())
report(section, "largest detectability value of white hypopigmentation, pustule and patch", benign_markers.max())

section = "Forced misses"
report(section, "forced misses", int(forced_miss.sum()))
report(section, "share of malignant images that are forced misses", forced_miss.sum() / n_mal)
for g in SIX:
    report(section, f"forced misses among {g} images", grp.forced_miss_share[g])
report(section, "share of forced misses with all 48 concepts", (grp.forced_miss_share_A48 * grp.images).sum() / n_mal)
per_profile = pd.Series(ids[forced_miss]).value_counts()
report(section, "share of forced misses held by the four largest such profiles", per_profile.iloc[:4].sum() / forced_miss.sum())
first = df[ids == per_profile.index[0]]
report(section, "malignant images in papule with brown hyperpigmentation", int(first.y.sum()))
for g, count in first[first.y == 1].malignant_group.value_counts().items():
    report(section, f"{g} images in papule with brown hyperpigmentation", count)
for dx in ["melanocytic nevi", "seborrheic keratosis", "dermatofibroma"]:
    report(section, f"{dx} in papule with brown hyperpigmentation", int((first[first.y == 0].dx == dx).sum()))
markers = df[["crust", "black", "telangiectasia", "ulcer"]].values.any(1)
report(section, "forced misses with crust, black, telangiectasia or ulcer", markers[forced_miss].mean())
report(section, "detected malignant images with one of these markers", markers[(y == 1) & ~forced_miss].mean())
report(section, "non-malignant images with one of these markers", markers[y == 0].mean())
n_forced = edits.image_id.nunique()
report(section, "forced misses rescued by at least one edit", rescued.image_id.nunique() / n_forced)
for c in ["crust", "black", "telangiectasia", "brown_hyperpigmentation"]:
    report(section, f"forced misses rescued by adding {c}", rescued[(rescued.edit == "add") & (rescued.concept == c)].image_id.nunique() / n_forced)
rescue_sets = {k: set(g.image_id) for k, g in rescued.groupby(["concept", "edit"])}
covered = set()
for step in range(1, 7):
    best = max(rescue_sets, key=lambda k: (len(rescue_sets[k] - covered), k))
    covered |= rescue_sets[best]
    report(section, f"forced misses rescued by the greedy selection after {step} edits ({best[1]} {best[0]})", len(covered) / n_forced)
lymphoma = edits[edits.malignant_group == "Cutaneous lymphoma"].image_id.nunique()
report(section, "cutaneous lymphoma forced misses rescued by brown hyperpigmentation",
       rescued[(rescued.malignant_group == "Cutaneous lymphoma") & (rescued.concept == "brown_hyperpigmentation")].image_id.nunique() / lymphoma)

section = "Sources of inconsistency"
report(section, "confirmed pairs with different concept vectors", counts["clusters_conflicting_concepts"])
for c in ["erythema", "plaque", "papule", "scale"]:
    report(section, f"Cohen's kappa of {c} across confirmed pairs", reliability.kappa[c])
embedding = complexity[complexity.input.str.startswith("image embeddings")].iloc[0]
annotated = complexity[complexity.input == "annotated concepts"].iloc[0]
report(section, "boundary images of the training split", embedding["boundary training images"])
report(section, "malignant boundary images of the training split", embedding["malignant"])
report(section, "grouped five-fold AUROC on image embeddings", embedding["grouped 5-fold AUROC"])
report(section, "grouped five-fold AUROC on annotated concepts", annotated["grouped 5-fold AUROC"])
report(section, "in-sample AUROC ceiling of the concepts on these images", annotated["in-sample concept ceiling AUC*"])
report(section, "mean cosine of different-label pairs in one profile", similarity.loc["same profile, different label", "mean cosine"])
report(section, "mean cosine of same-label pairs in one profile", similarity.loc["same profile, same label", "mean cosine"])

section = "Filtering"
ft = filter_training.set_index("filter")
fo = filter_oof.set_index("filter")
report(section, "training gamma after tau-editing", ft.train_gamma["tau"])
report(section, "out-of-fold AUROC without filter", fo.auroc["none"])
report(section, "lowest out-of-fold AUROC after a filter", fo.auroc.drop("none").min())
report(section, "highest out-of-fold AUROC after a filter", fo.auroc.drop("none").max())
report(section, "lowest AUROC on filtered test folds", ft.filtered_cv_auroc.drop("none").min())
report(section, "highest AUROC on filtered test folds", ft.filtered_cv_auroc.drop("none").max())
report(section, "AUROC of majority editing on filtered test folds", ft.filtered_cv_auroc["majority"])
report(section, "out-of-fold AUROC of majority editing", fo.auroc["majority"])
marker_u = U[:, [E.index(c) for c in ["crust", "black", "telangiectasia", "ulcer"]]].any(1)[ids_u]
called_u = {h: maps[h][ids_u] >= 0.5 for h in FILTERS}
for h in ["none", "symmetric", "majority", "asymmetric"]:
    report(section, f"images in marker profiles called malignant, {h}", called_u[h][marker_u].mean())
    report(section, f"images in other profiles called malignant, {h}", called_u[h][~marker_u].mean())
mixed_u = (malignant_u > 0) & (malignant_u < images_u)
rate_u = malignant_u / images_u
for h in ["none", "asymmetric"]:
    report(section, f"Pearson r between score and rate of inconsistent profiles, {h}", np.corrcoef(rate_u[mixed_u], maps[h][mixed_u])[0, 1])
ret = pd.read_csv(OUT / "filter_retention.csv").pivot(index="group", columns="filter", values="retained")
for h in ["symmetric", "majority", "weighted"]:
    report(section, f"malignant training images kept, {h}", ft.train_malignant_kept[h])
    report(section, f"share of malignant training images kept, {h}", ft.train_malignant_kept[h] / ft.train_malignant_kept["none"])
report(section, "malignant training images", ft.train_malignant_kept["none"])
report(section, "share of non-malignant training images kept, majority", ret.loc["Non-malignant", "majority"])
for g in ["BCC", "SCC", "Cutaneous lymphoma"]:
    report(section, f"share of {g} training images kept, symmetric", ret.loc[g, "symmetric"])
points = pd.DataFrame([{"kept": ret.loc[g, h], "sensitivity": fo.loc[h, g]} for h in FILTERS for g in SIX])
report(section, "Pearson r between retained share and sensitivity", np.corrcoef(points.kept, points.sensitivity)[0, 1])
report(section, "points in that correlation", len(points))
for h in FILTERS:
    report(section, f"missed malignant images, {h}", round((1 - fo.sensitivity[h]) * n_mal))
report(section, "specificity of asymmetric removal", fo.specificity["asymmetric"])
report(section, "sensitivity of asymmetric removal", fo.sensitivity["asymmetric"])
f, t_, _ = roc_curve(y, oof["none"])
report(section, "specificity of the unfiltered model at the same sensitivity", 1 - f[t_ >= fo.sensitivity["asymmetric"]].min())

section = "Concept bottleneck models"
for m in MODELS:
    report(section, f"test AUROC, {m}", summary.auroc_mean[m])
report(section, "test AUROC of a logistic regression on annotated concepts", baselines.value[0])
report(section, "AUROC ceiling of the annotated test concepts", baselines.value[1])
concept_rows = metrics[(metrics.model == "concepts") & (metrics.split == "test") & (metrics.source == "all")]
report(section, "test concept AUROC, mean", concept_rows.concept_macro_auroc.mean())
report(section, "test concept AUROC, standard deviation", concept_rows.concept_macro_auroc.std())
report(section, "test images with the whole profile predicted correctly", concept_rows.exact_profile_rate.mean())
report(section, "AUROC gain of the soft model over the hard model", summary.auroc_mean["soft"] - summary.auroc_mean["hard"])
report(section, "AUROC gain of the hybrid model over the hard model", summary.auroc_mean["hybrid"] - summary.auroc_mean["hard"])
hybrid_rows = metrics[(metrics.model == "hybrid") & (metrics.split == "test") & (metrics.source == "all")].set_index("seed")
report(section, "share of the hybrid logit carried by the visual units", hybrid_rows.visual_share.mean())
report(section, "test decisions changed by the visual units", hybrid_rows.flipped.mean())
for s in (42, 64, 128):
    h = hybrid[hybrid.seed == s]
    report(section, f"seed {s}, visual weight norm", hybrid_rows.w_l1[s])
    report(section, f"seed {s}, decisions changed", int(h.changed.sum()))
    for label, cls, up in [("malignant images rescued", 1, True), ("malignant images lost", 1, False),
                           ("false positives added", 0, True), ("false positives removed", 0, False)]:
        report(section, f"seed {s}, {label}", int((h.changed & (h.y_true == cls) & ((h.logit > 0) == up)).sum()))

section = "Missed malignancies"
mr = miss_rates.set_index("malignant_group")
n_test = int(mr.test_images["All malignant"])
report(section, "test malignancies", n_test)
for g in ["All malignant"] + SIX:
    report(section, f"lowest miss rate over the models, {g}", mr.loc[g, MODELS].min())
    report(section, f"highest miss rate over the models, {g}", mr.loc[g, MODELS].max())
for g in SIX:
    report(section, f"test images, {g}", mr.test_images[g])
for m in MODELS:
    report(section, f"missed test malignancies, {m}", (1 - summary.sensitivity_mean[m]) * n_test)
mean_outcomes = outcomes.groupby("model").mean(numeric_only=True)
report(section, "forced cases among the test malignancies", int(mean_outcomes.forced_missed.iloc[0] + mean_outcomes.forced_detected.iloc[0]))
for m in MODELS:
    o = mean_outcomes.loc[m]
    report(section, f"forced cases missed, {m}", o.forced_missed)
    report(section, f"forced cases detected, {m}", o.forced_detected)
    report(section, f"other malignancies missed, {m}", o.other_missed)
    report(section, f"share of the misses that are forced cases, {m}", o.forced_missed / (o.forced_missed + o.other_missed))
cls = np.where(test.y_true == 0, "non-malignant", np.where(test.forced, "forced", "other malignant"))
medians = test.assign(cls=cls).groupby(["model", "cls"]).logit.median().unstack()
for m in MODELS:
    for c in medians.columns:
        report(section, f"median logit, {c}, {m}", medians.loc[m, c])
op = operating_points.groupby(["target", "model"])[["sensitivity", "specificity"]].mean()
for target in (0.9, 0.95):
    for m in MODELS:
        report(section, f"test sensitivity at the {target:.0%} target, {m}", op.loc[(target, m), "sensitivity"])
        report(section, f"test specificity at the {target:.0%} target, {m}", op.loc[(target, m), "specificity"])
        report(section, f"missed test malignancies at the {target:.0%} target, {m}", (1 - op.loc[(target, m), "sensitivity"]) * n_test)
flag_rate = flagged.groupby("diagnosis").flag_rate.mean()
for dx in ["seborrheic keratosis", "pyogenic granuloma", "neutrophilic dermatoses", "melanocytic nevi"]:
    report(section, f"flag rate of {dx} at the 95% target", flag_rate[dx])

section = "Model selection"
stage_a = pd.read_csv(COLAB / "stageA_rung2.csv")
stage_b = pd.read_csv(COLAB / "stageB_heads.csv")
small = stage_b[(stage_b.model == "hybrid") & (stage_b.l1 < 0.1)]
selected = pd.read_csv(COLAB / "selected_heads.csv").set_index("model")
report(section, "best validation concept AUROC after 30 epochs", stage_a.valid_concept_macro_auroc.max())
report(section, "best validation AUROC of hybrid heads with small penalties", small.valid_auroc.max())
report(section, "smallest visual share of hybrid heads with small penalties", small.visual_share.min())
report(section, "selected number of visual units", int(selected.n_visual["hybrid"]))
report(section, "selected L1 weight", selected.l1["hybrid"])

reported = pd.DataFrame(reported, dtype=object)
reported.to_csv(OUT / "reported_values.csv", index=False)
with pd.option_context("display.max_rows", 400, "display.max_colwidth", 90):
    display(reported)

,section,quantity,value
0,Clean dataset,Fitzpatrick17k images annotated by SkinCon,3690
1,Clean dataset,DDI images annotated by SkinCon,656
2,Clean dataset,Fitzpatrick17k images flagged by the annotators,460
3,Clean dataset,DDI images flagged by the annotators,20
4,Clean dataset,Fitzpatrick17k images excluded by quality control,3
5,Clean dataset,images entering the cleaning rules,3863
6,Clean dataset,candidate near-copy pairs,475
7,Clean dataset,confirmed duplicate pairs,30
8,Clean dataset,confirmed pairs that satisfy both detectors,26
9,Clean dataset,pairs with cosine of at least 0.9,453
